In [ ]:
# Credit Risk Modeling — Stage 1: Data Preparation

## Basel II–Aligned PD Scorecard Pipeline | LendingClub Consumer Loan Data

**Project:** End-to-End Credit Risk Model (PD / LGD / EAD / Expected Loss)
**Stage:** 1 of 5 — Data Preparation
**Dataset:** LendingClub Loan Data (2007–2018)
**Author:** Harsh
**Framework:** Basel II Internal Ratings-Based (IRB) Approach

---

# Executive Summary

This notebook establishes a production-oriented, model-ready dataset for Probability of Default (PD) estimation using LendingClub consumer loan data.

The objective is not merely to clean data, but to construct a causally valid underwriting dataset aligned with real-world banking model development standards. The preprocessing pipeline is specifically designed to prevent target leakage, preserve temporal consistency, and maintain economic interpretability for downstream scorecard modeling.

The pipeline includes:

* Binary target definition aligned with Basel-style default modeling
* Strict removal of post-origination leakage variables
* Time-based Train / Test / Out-of-Time splitting
* Economic feature engineering
* Sparse feature elimination
* Economically driven missing value treatment
* Controlled outlier handling (winsorization)
* Feature-space stabilization for WOE transformation

The final outputs are six modeling datasets:

* `X_train`
* `X_test`
* `X_oot`
* `y_train`
* `y_test`
* `y_oot`

These datasets are suitable for downstream:

* WOE transformation
* Information Value analysis
* Logistic regression scorecard modeling
* Basel-style PD estimation

---

# Business Context

In retail banking credit risk modeling, the primary objective is to estimate:

[
PD = P(\text{Default} \mid \text{Borrower Characteristics})
]

A critical requirement is that the model must only learn from information available at the time of underwriting.

Variables generated after loan origination create target leakage and invalidate model performance.

Examples:

* `total_pymnt` reflects realized repayments after loan issuance
* `recoveries` exists only after default
* `settlement_status` directly embeds post-default behavior

Including such variables causes the model to “cheat” by learning future outcomes rather than borrower risk.

Therefore, Stage 1 focuses primarily on:

* causal integrity
* temporal consistency
* deployability under real underwriting conditions

---

# Methodology Overview

The preprocessing framework follows a structured risk modeling workflow:

1. Data Loading & Inspection
2. Target Definition
3. Time Variable Construction
4. Leakage & Future Information Removal
5. Temporal Train/Test/OOT Split
6. Feature Structuring
7. Credit History Transformation
8. Sparse Feature Elimination
9. Missing Value Treatment
10. Outlier Treatment (Winsorization)

Each preprocessing step is implemented with explicit economic rationale rather than generic machine learning heuristics.

---

# Step 1 — Load Data and Initial Inspection

```python
import numpy as np
import pandas as pd

pd.options.display.max_columns = None

df_raw = pd.read_csv(r'loan.csv', low_memory=False)

df1 = df_raw.copy()

print("Shape:", df1.shape)
print(df1['loan_status'].value_counts(normalize=True))
```

Additional diagnostics are performed:

* missing value ratios
* datatype inspection
* class imbalance review
* yearly loan distribution analysis

Initial inspection confirms a typical retail credit portfolio imbalance where defaults represent a minority class.

This is expected in banking datasets and later influences:

* score calibration
* threshold selection
* validation metrics

---

# Step 2 — Target Definition (Good vs Bad)

```python
good_status = [
    'Fully Paid',
    'Does not meet the credit policy. Status:Fully Paid'
]

bad_status = [
    'Charged Off',
    'Default',
    'Does not meet the credit policy. Status:Charged Off'
]

df1 = df1[df1['loan_status'].isin(good_status + bad_status)].copy()

df1['target'] = np.where(df1['loan_status'].isin(bad_status), 0, 1)
```

Interpretation:

* `target = 1` → Non-default
* `target = 0` → Default

This follows scorecard convention where higher scores correspond to lower credit risk.

Economically:

* “Charged Off” implies realized loss
* “Fully Paid” implies survival through loan maturity

---

# Step 3 — Time Variable Construction

```python
df1['issue_d_dt'] = pd.to_datetime(
    df1['issue_d'],
    format='%b-%Y',
    errors='coerce'
)
```

Time analysis is mandatory in credit modeling because default behavior changes across economic cycles.

The pipeline evaluates:

* yearly loan volumes
* yearly default rates
* portfolio growth trends
* temporal drift

A visualization is generated to inspect cumulative portfolio growth and default rate behavior across vintages.

---

# Step 4 — Leakage Removal (Critical Stage)

The pipeline removes variables containing post-origination information.

## Leakage Variables

```python
leak_cols = [
    'out_prncp','out_prncp_inv',
    'total_pymnt','total_pymnt_inv',
    'total_rec_prncp','total_rec_int',
    'total_rec_late_fee',
    'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d',
    'last_pymnt_amnt',
    'next_pymnt_d'
]
```

## Future-State Variables

```python
future_cols = [
    'last_credit_pull_d',
    'hardship_flag',
    'hardship_type',
    'hardship_reason',
    'hardship_status',
    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term'
]
```

## Non-Predictive Administrative Variables

```python
drop_cols = [
    'id','member_id','url','desc',
    'title','zip_code',
    'emp_title','pymnt_plan'
]
```

Interpretation:

This is the most important stage in the entire preprocessing pipeline.

Example:

```python
recoveries > 0
```

already implies default realization.

If retained, the model would artificially achieve unrealistically high predictive power while failing in production deployment.

The resulting dataset now reflects only underwriting-time information.

---

# Step 5 — Time-Based Dataset Split

The dataset is sorted chronologically before splitting.

```python
train = df1[df1['issue_d_dt'] < '2014-01-01']
test  = df1[
    (df1['issue_d_dt'] >= '2014-01-01') &
    (df1['issue_d_dt'] < '2015-01-01')
]
oot   = df1[df1['issue_d_dt'] >= '2015-01-01']
```

Outputs:

* Train → model estimation
* Test → validation
* OOT (Out-of-Time) → temporal robustness evaluation

OOT validation is essential in banking because models must generalize across changing economic conditions, not merely fit historical data.

Target distributions are evaluated across all splits to monitor population drift and bad-rate stability.

---

# Step 6 — Feature Filtering & Sparse Variable Elimination

The raw LendingClub dataset contains many:

* highly sparse variables
* duplicated signals
* post-2015 engineered features
* thin-file attributes
* secondary applicant fields

Several low-quality or unstable variables are removed to improve:

* model stability
* WOE monotonicity
* scorecard interpretability
* multicollinearity behavior

Examples include:

```python
drop_features = [
    'all_util',
    'bc_util',
    'total_bc_limit',
    'avg_cur_bal',
    'tot_cur_bal'
]
```

Secondary applicant variables with excessive missingness are also removed.

Columns with >90% missing values are automatically identified and dropped.

Interpretation:

This prevents unstable binning behavior and reduces overfitting risk in downstream scorecard development.

---

# Step 7 — Feature Engineering

## Loan Term Transformation

```python
d['term_int'] = d['term'].str.replace(
    ' months',''
).astype(float)
```

Interpretation:

Loan term directly affects:

* installment burden
* repayment horizon
* default exposure duration

---

## Employment Length Transformation

```python
df['emp_length_clean']
```

is converted into numeric employment tenure.

Examples:

* `< 1 year` → 0
* `10+ years` → 10

Interpretation:

Employment tenure acts as a proxy for:

* income stability
* labor-market consistency
* repayment capacity

Longer employment histories generally correlate with lower default risk.

---

# Step 8 — Credit History Transformation

```python
d['earliest_cr_line_dt'] = pd.to_datetime(
    d['earliest_cr_line'],
    format='%b-%Y',
    errors='coerce'
)
```

Credit history age is then calculated:

```python
mths_since_earliest_cr_line
```

Interpretation:

Longer credit histories reduce uncertainty regarding borrower behavior.

Negative values caused by data quality issues are corrected to zero.

Intermediate date columns are removed after transformation to avoid redundancy.

---

# Step 9 — Missing Value Treatment (Economic Logic Driven)

The missing value strategy is intentionally economics-based rather than statistically naive.

---

## 9.1 Categorical Variables

All categorical missing values are filled with:

```python
'MISSING'
```

This preserves missingness as an explicit risk segment.

---

## 9.2 Time-Based Variables → 999

Examples:

```python
mths_since_last_delinq
mths_since_recent_revol_delinq
```

Interpretation:

Missing values often indicate absence of delinquency events.

Thus:

```python
999
```

acts as a behavioral signal meaning:

* “No delinquency observed”

rather than unknown data.

---

## 9.3 Structural Missing Variables → -1

Examples:

```python
total_rev_hi_lim
tot_hi_cred_lim
num_bc_tl
```

Interpretation:

Missingness frequently indicates:

* thin credit file
* unavailable revolving history
* lack of bureau records

Therefore:

```python
-1
```

creates a separate structural risk category.

---

## 9.4 Utilization Edge Variables → -1

Examples:

```python
bc_open_to_buy
percent_bc_gt_75
```

These often become missing due to absent revolving trade lines.

---

## 9.5 Remaining Numeric Variables → Median

For standard numeric fields:

```python
median = X_train[col].median()
```

Median imputation is applied using only training-set statistics to prevent data leakage.

---

# Step 10 — Winsorization (Outlier Treatment)

Extreme values distort:

* logistic regression coefficients
* WOE stability
* monotonic binning
* score scaling behavior

Selected variables are winsorized at the 1st and 99th percentiles.

```python
winsor_cols = [
    'annual_inc',
    'loan_amnt',
    'installment',
    'revol_bal',
    'open_acc',
    'total_acc',
    'dti',
    'int_rate'
]
```

Interpretation:

Extremely large incomes or revolving balances do not reduce risk linearly and can destabilize scorecard calibration.

Winsorization preserves monotonic relationships while limiting tail distortion.

---

# Final Outputs

The final datasets contain:

* no target leakage
* temporally valid splits
* economically consistent missing value treatment
* engineered underwriting features
* stabilized numeric distributions

Outputs:

```python
X_train
X_test
X_oot

y_train
y_test
y_oot
```

At this stage:

* the dataset reflects real underwriting conditions
* variables are aligned with credit risk theory
* the feature space is ready for WOE transformation

Most importantly:

the model will now learn borrower risk characteristics rather than realized loan outcomes.

---

# Limitations

Despite robust preprocessing, several limitations remain:

## 1. Point-in-Time Bias

No macroeconomic variables are incorporated:

* unemployment
* GDP growth
* interest-rate regime
* inflation cycle

Therefore, calibration reflects historical conditions only.

---

## 2. Reject Inference Bias

Only approved applicants are observed.

Rejected borrowers remain unmodeled, introducing selection bias.

---

## 3. Limited Behavioral Dynamics

The dataset focuses primarily on application-stage variables rather than longitudinal repayment behavior.

---

## 4. Static Snapshot Modeling

Borrower evolution over time is not modeled dynamically.

---

## 5. Portfolio-Specific Calibration

The model is calibrated specifically on LendingClub consumer lending behavior and may require recalibration for other portfolios.

---

# Transition to Stage 2 — Feature Engineering

This concludes Stage 1: Data Preparation.

The resulting datasets are:

* causally sound
* temporally consistent
* economically interpretable
* scorecard-ready

Stage 2 will implement:

* WOE transformation
* Information Value analysis
* fine classing
* coarse classing
* monotonicity validation
* multicollinearity assessment (VIF)

to produce the final modeling feature set for Basel-aligned logistic regression scorecards.

**Next Notebook:** `02_feature_engineering.ipynb`
